## Bonus: Llama-3 Fine-Tuning with QLoRA

**What is QLoRA?**
Instead of updating all 8 billion parameters (which needs 80GB+ VRAM), LoRA adds small trainable adapter layers alongside the frozen model weights. QLoRA additionally quantizes the base model to 4-bit to save memory. Result: fine-tuning a large LLM on a single Colab GPU.

**Tools:** HuggingFace PEFT, TRL, bitsandbytes, wandb
**AI used:** Claude (Anthropic)
**W&B:** https://wandb.ai/

```
pip install transformers peft trl bitsandbytes accelerate wandb scikit-learn
```

> Accept the Llama-3 license at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
> Add your HuggingFace token to Colab Secrets as HF_TOKEN

In [ ]:
!pip install -U transformers trl peft accelerate bitsandbytes datasets

## 0 — Imports

In [ ]:
import json
import random
import matplotlib.pyplot as plt
import wandb
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score, classification_report
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1 — Load Data

In [ ]:
random.seed(42)

train_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  remove=('headers', 'footers', 'quotes'))
CLASS_NAMES = train_data.target_names

# Use a subset for fine-tuning (full dataset is slow on Colab)
N_TRAIN = 2000
N_EVAL  = 200

train_indices = random.sample(range(len(train_data.data)), N_TRAIN)
eval_indices  = random.sample(range(len(test_data.data)),  N_EVAL)

print(f'Fine-tuning on {N_TRAIN} samples, evaluating on {N_EVAL}')

## 2 — Format Data as Instruction Prompts

LLMs are fine-tuned using instruction-following format. Each training example becomes:
- **Input:** text + list of classes
- **Output:** the correct class name

In [ ]:
CLASS_LIST = '\n'.join(f'- {name}' for name in CLASS_NAMES)

def format_example(text, label):
    return (
        f"Classify the following text into one of these categories:\n{CLASS_LIST}\n\n"
        f"Text: {text[:300]}\n\nCategory: {CLASS_NAMES[label]}"
    )

train_examples = [
    {'text': format_example(train_data.data[i], train_data.target[i])}
    for i in train_indices
]

hf_dataset = Dataset.from_list(train_examples)
print(f'Dataset ready. Example:\n{train_examples[0]["text"][:300]}...')

## 3 — Load Model + Apply LoRA

**LoRA settings:**
- `r=8`: rank of the adapter matrices — higher = more parameters = more capacity
- `lora_alpha=16`: scaling factor for the LoRA updates
- `target_modules`: which layers to add LoRA to (query and value projections in attention)
- `lora_dropout=0.05`: regularization

In [ ]:
MODEL_ID   = 'meta-llama/Meta-Llama-3-8B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print('Loading Llama-3...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto'
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# This will show something like: trainable params: 3,407,872 || all params: 8,033,669,120
# Only ~0.04% of parameters are trained — that's the power of LoRA

In [ ]:
import transformers
import trl
import peft
import accelerate
import bitsandbytes

print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

## 4 — Fine-Tune with SFTTrainer

In [ ]:
wandb.init(
    project='nalapro-20newsgroups',
    name='llama3-qlora',
    config={
        'n_train': N_TRAIN,
        'lora_r': 8,
        'lora_alpha': 16,
        'epochs': 1,
        'batch_size': 4
    }
)

sft_config = SFTConfig(
    output_dir='nalapro/llama3_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    logging_steps=10,
    save_strategy='no',
    report_to='wandb',
    dataset_text_field='text',
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=hf_dataset,
)

print('Starting QLoRA fine-tuning...')
trainer.train()

wandb.finish()

print('Fine-tuning complete.')

## 5 — Evaluate Fine-Tuned Model

In [ ]:
eval_texts  = [test_data.data[i]   for i in eval_indices]
eval_labels = [test_data.target[i] for i in eval_indices]

def ask_llama(prompt):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=20,
                                do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def match_class(response):
    response = response.lower()
    for i, name in enumerate(CLASS_NAMES):
        if name.lower() in response:
            return i
    return 0

def eval_prompt(text):
    return (
        f"Classify the following text into one of these categories:\n{CLASS_LIST}\n\n"
        f"Text: {text[:300]}\n\nCategory:"
    )

model.eval()
preds = []
for i, text in enumerate(eval_texts):
    response = ask_llama(eval_prompt(text))
    preds.append(match_class(response))
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{N_EVAL} done...')

acc_lora = accuracy_score(eval_labels, preds)
print(f'\nLlama-3 QLoRA Accuracy: {acc_lora:.4f}')
print(classification_report(eval_labels, preds, target_names=CLASS_NAMES, zero_division=0))

## 6 — Full Comparison Including Bonus

In [ ]:
# Paste in all results
acc_w2v   = 0.586
acc_tfidf = 0.636
acc_lsa   = 0.615
acc_bert     = 0.698
acc_bert_mlm = 0.710
acc_zero     = 0.255
acc_few      = 0.430

labels = ['Word2Vec', 'TF-IDF', 'TF-IDF+LSA', 'BERT', 'BERT+MLM',
          'Llama3\nZero-Shot', 'Llama3\nFew-Shot', 'Llama3\nQLoRA']
accs   = [acc_w2v, acc_tfidf, acc_lsa, acc_bert, acc_bert_mlm, acc_zero, acc_few, acc_lora]
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#9467BD','#8C564B','#E377C2','#17BECF']

plt.figure(figsize=(14, 5))
bars = plt.bar(labels, accs, color=colors, width=0.6)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{acc:.3f}', ha='center', fontweight='bold')
plt.ylim(0, 1.05)
plt.ylabel('Test Accuracy')
plt.title('Final Comparison: All Methods Including Bonus', fontsize=14)
plt.tight_layout()
plt.savefig('final_comparison_bonus.png', dpi=150)
plt.show()

## 7 — Save Results

In [ ]:
from google.colab import files

with open('lora_results.json', 'w') as f:
    json.dump({'qlora_accuracy': acc_lora}, f)

files.download('lora_results.json')
files.download('final_comparison_bonus.png')
print('Done.')